In [1]:
import pandas as pd
from sklearn.model_selection import StratifiedKFold

from src.model import Model
from src.simulation import simulate_patients
from src.utils import get_labelled_sequences


# minimum percentage of time that the patient must have worn the device over a given week
time_worn_threshold = 0.7

# glucose threshold below which we detect the onset of hypoglycemia, in mg/dL
glucose_threshold = 54

# minimum length of a hypoglycemic event, in minutes
event_duration_threshold = 15


In [2]:

# generate a dummy dataset
data = simulate_patients(
    freq=5,      # sampling frequency of the time series, in minutes
    length=45,   # length of the time series, in days
    num=100,     # number of time series
)

In [4]:
data

,id,ts,gl
0,0,2024-07-28 00:00:00,141.856524
1,0,2024-07-28 00:05:00,142.107532
2,0,2024-07-28 00:10:00,142.840166
3,0,2024-07-28 00:15:00,144.381167
4,0,2024-07-28 00:20:00,146.943918
...,...,...,...
1295995,99,2024-09-10 23:35:00,166.160151
1295996,99,2024-09-10 23:40:00,168.393125
1295997,99,2024-09-10 23:45:00,169.810047
1295998,99,2024-09-10 23:50:00,NaN


In [54]:
# reshape the dataset from long to wide
data = data.pivot(index='ts', columns=['id'], values=['gl'])
data.columns = data.columns.get_level_values(level='id')
# split the dataset into sequences
sequences = get_labelled_sequences(
    data=data,
    time_worn_threshold=time_worn_threshold,
    glucose_threshold=glucose_threshold,
    event_duration_threshold=event_duration_threshold,
)

In [56]:
# split the sequences into folds
skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

# create a list for storing the results for each fold
results = []

# loop across the folds
for i, (train_index, test_index) in enumerate(skf.split(X=[s['X'] for s in sequences], y=[s['Y'] for s in sequences])):
    
    # fit the model to the training set
    model = Model()

    model.fit(
        sequences=[sequences[i] for i in train_index],
        sequence_length=int(7 * 24 * 60 // 5),
        l1_penalty=0.005,
        l2_penalty=0.05,
        learning_rate=0.00001,
        batch_size=32,
        epochs=1000,
        seed=42,
        verbose=0
    )

    # evaluate the model on the test set
    metrics = model.evaluate(sequences=[sequences[i] for i in test_index])

    # save the results
    results.append(metrics)

# organize the results in a data frame
results = pd.DataFrame(results)

# average the results
print(results.mean())

accuracy             0.950000
balanced_accuracy    0.927884
precision            0.835588
sensitivity          0.894444
specificity          0.961324
f1                   0.860186
auc                  0.987628
dtype: float64
